# PPE Detection: training on Kaggle

Kaggle version of `train_colab.ipynb`, with the same data preparation, training settings and evaluation. Kaggle gives about 30 free GPU hours a week and 12-hour sessions, enough for all three runs in one go.

**Setup (once):** in the notebook editor, open **Settings** and set **Accelerator → GPU T4 x2** and **Internet → On** (both need a phone-verified Kaggle account). The 4 datasets are attached as inputs; if one is missing, `kagglehub` downloads it.

**Run:** **Save Version → Save & Run All (Commit)**. You can close the browser while it runs.

What it does:
1. Merges the 4 Kaggle datasets into one 8-class YOLO dataset (`prepare_data.py`).
2. Trains a **baseline** YOLOv8s and a **balanced** YOLOv8s (repeat-factor sampling for goggles, no-goggles, no-mask).
3. Trains **YOLOv5s** on the same split for a model comparison.
4. Evaluates every run on the held-out test split.

Everything is saved under `/kaggle/working/ppe-detection`, which appears in the notebook's **Output** tab when the run finishes.

In [ ]:
# Settings (same as train_colab.ipynb)
EPOCHS = 25          # about 1.5-2 h per run on a T4
MODEL = "yolov8s.pt"
RUN_YOLOV5 = True    # extra YOLOv5 vs YOLOv8 comparison run (another ~1.5-2 h)
REPO = "https://github.com/NishaadG/ppe-detection.git"

In [ ]:
!nvidia-smi -L
import os, shutil
WORK = '/kaggle/working/ppe-detection'   # kept as notebook output
TMP = '/tmp'                             # raw data, merged dataset and code (not kept)
os.makedirs(WORK, exist_ok=True)

In [ ]:
%cd {TMP}
!rm -rf ppe-detection && git clone -q {REPO}
%cd {TMP}/ppe-detection
!pip -q install "ultralytics==8.3.94" kagglehub

In [ ]:
# Attached inputs resolve to /kaggle/input; otherwise kagglehub downloads them.
# They are copied (not linked) so prepare_data.py sees ordinary folders under raw/.
import kagglehub
DATASETS = {
    "helmet": "andrewmvd/hard-hat-detection",
    "mask": "andrewmvd/face-mask-detection",
    "vest": "adilshamim8/safety-vests-detection-dataset",
    "goggles": "ketakichalke/ppe-kit-detection-construction-site-workers",
}
for name, slug in DATASETS.items():
    dst = f'{TMP}/raw/{name}'
    if not os.path.exists(dst):
        src = kagglehub.dataset_download(slug)
        shutil.copytree(src, dst)
    print(f'{name:8s}', sorted(os.listdir(dst))[:6])
!python prepare_data.py --raw {TMP}/raw --out {TMP}/data --seed 42
!cp {TMP}/data/stats.json {TMP}/data/class_distribution.png {WORK}/
from IPython.display import Image, display
display(Image(f'{TMP}/data/class_distribution.png'))

The class names printed above for `vest` and `goggles` were checked against `CSV_MAP` / `YOLO_MAP` in `prepare_data.py` (vest: "Safety Vest", "NO-Safety Vest"; goggles ids 0 Helmet, 2 Vest, 4 Goggles, 7 no_helmet, 8 no_goggle).

In [ ]:
from ultralytics import YOLO

def train(name, data, model=MODEL, epochs=EPOCHS):
    if os.path.exists(f'{WORK}/runs/{name}/weights/best.pt'):
        print(f'{name}: already trained')
        return
    YOLO(model).train(data=data, epochs=epochs, imgsz=640, batch=16, seed=42, patience=10,
                      workers=2, project=f'{WORK}/runs', name=name, exist_ok=True, plots=True)

In [ ]:
train('baseline', f'{TMP}/data/ppe.yaml')

In [ ]:
train('balanced', f'{TMP}/data/ppe_balanced.yaml')

In [ ]:
if RUN_YOLOV5:
    train('yolov5s', f'{TMP}/data/ppe.yaml', model='yolov5su.pt')

In [ ]:
runs = [f'{WORK}/runs/{n}' for n in ('baseline', 'balanced', 'yolov5s') if os.path.exists(f'{WORK}/runs/{n}/weights/best.pt')]
runs_arg = ' '.join(runs)
!python evaluate.py --data {TMP}/data/ppe.yaml --runs {runs_arg} --out {WORK}/results

In [ ]:
best = open(f'{WORK}/results/best_run.txt').read().strip()
os.makedirs(f'{WORK}/weights', exist_ok=True)
shutil.copy(f'{WORK}/runs/{best}/weights/best.pt', f'{WORK}/weights/best.pt')
print('Best run:', best, '-> copied to', f'{WORK}/weights/best.pt')
for f in ('confusion_matrix_normalized.png', 'PR_curve.png'):
    p = f'{WORK}/results/{best}_test/{f}'
    if os.path.exists(p):
        display(Image(p, width=700))

In [ ]:
# A few test-set predictions, saved for the README
import glob, random
random.seed(0)
imgs = random.sample(sorted(glob.glob(f'{TMP}/data/images/test/*')), 6)
model = YOLO(f'{WORK}/weights/best.pt')
os.makedirs(f'{WORK}/samples', exist_ok=True)
for i, p in enumerate(imgs):
    r = model.predict(p, conf=0.35, verbose=False)[0]
    out = f'{WORK}/samples/sample_{i}.jpg'
    r.save(filename=out)
    display(Image(out, width=420))

## Done
Download the results from the notebook's **Output** tab (or run `kaggle kernels output <user>/<notebook> -p out`). They are all in `ppe-detection/`:
- `results/comparison.md` (the table for the README and resume)
- `stats.json`
- `weights/best.pt` (goes into the Hugging Face Space)
- `samples/` and `results/<best>_test/` plots (for the README)